# 06 — Final Training, Evaluation & Export (Stage 2)

**Purpose:** Train the best configuration identified from ablation studies
(notebooks 04-05), run comprehensive evaluation, and export the model
for deployment.

**Outputs:**
- Final checkpoint (`.pth`)
- ONNX export (`.onnx`)
- TorchScript export (`.pt`)
- Comprehensive evaluation report (JSON + plots)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless tensorboard onnx onnxruntime

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/EcoCAR-Perception-Pipeline-YOLO26-BDD100K/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:
import torch
from lib.config import cfg
from lib.models import get_net

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg.defrost()
cfg.DATASET.DATAROOT = '/content/bdd100k/images/100k'
cfg.DATASET.LABELROOT = '/content/bdd100k/labels/100k'
cfg.DATASET.LANEROOT = '/content/bdd100k/lane_masks'
cfg.DRIVE.CHECKPOINT_DIR = '/content/drive/MyDrive/EcoCAR/checkpoints'
cfg.freeze()

# Load best model
model = get_net(cfg).to(device)
model.gr = 1.0
model.nc = 5
model.names = cfg.MODEL.VEHICLE_CLASSES

ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth')
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]}')

In [ ]:
# ── ONNX Export ──
export_dir = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)

dummy_input = torch.randn(1, 3, 640, 640).to(device)

onnx_path = os.path.join(export_dir, 'vehicle_lane.onnx')
torch.onnx.export(
    model, dummy_input, onnx_path,
    opset_version=12,
    input_names=['images'],
    output_names=['det_out', 'lane_seg'],
    dynamic_axes={
        'images': {0: 'batch'},
        'det_out': {0: 'batch'},
        'lane_seg': {0: 'batch'},
    }
)
print(f'ONNX exported to {onnx_path}')

# Validate ONNX
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print('ONNX model validation passed!')

In [ ]:
# ── TorchScript Export ──
ts_path = os.path.join(export_dir, 'vehicle_lane.pt')
traced = torch.jit.trace(model, dummy_input)
traced.save(ts_path)
print(f'TorchScript exported to {ts_path}')

# Verify
loaded = torch.jit.load(ts_path, map_location=device)
with torch.no_grad():
    out = loaded(dummy_input)
print(f'TorchScript verification: det={type(out[0])}, lane_seg={out[1].shape}')

In [ ]:
# ── Model size summary ──
param_count = sum(p.numel() for p in model.parameters())
model_size_mb = os.path.getsize(ckpt_path) / 1024 / 1024
onnx_size_mb = os.path.getsize(onnx_path) / 1024 / 1024

print(f'Parameters:     {param_count/1e6:.2f}M')
print(f'Checkpoint:     {model_size_mb:.1f} MB')
print(f'ONNX:           {onnx_size_mb:.1f} MB')
print(f'Export dir:     {export_dir}')